# AI Study Buddy — One-Click Google Colab Fine-Tuner

This notebook is **100% self-contained**. It automatically downloads open datasets (`allenai/sciq`), fine-tunes **FLAN-T5** models for text summarization and academic paraphrasing on GPU, and saves model weights for deployment.

### How to Run
1. In Google Colab, select **Runtime -> Change runtime type -> T4 GPU**.
2. Run all cells in sequence (`Runtime -> Run all` or Shift+Enter per cell).

### Step 1: Check GPU & Install Dependencies

In [ ]:
import os
import sys
import subprocess

# Verify GPU
os.system("nvidia-smi")

# Install fine-tuning libraries
packages = ["transformers", "datasets", "evaluate", "rouge_score", "sacrebleu", "accelerate", "huggingface_hub"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

### Step 2: Download Open Dataset (SciQ) & Generate Training Data

In [ ]:
import json
from datasets import load_dataset

print("Loading allenai/sciq open dataset...")
sciq = load_dataset("allenai/sciq", split="train")

sum_records = []
para_records = []

for idx, sample in enumerate(sciq):
    if idx >= 400:
        break
    q = (sample.get("question") or "").strip()
    ans = (sample.get("correct_answer") or "").strip()
    supp = (sample.get("support") or "").strip()
    if not q or not ans or not supp:
        continue

    # Summarization pair
    sum_records.append({
        "input": f"Summarize the following study material concisely:\n\nContext: {supp}\nQuestion: {q}",
        "output": f"Answer: {ans}. Explanation: {supp}"
    })
    # Paraphrasing pair
    para_records.append({
        "input": f"Paraphrase the following sentence in a formal academic style:\n\nExplain: {q}",
        "output": f"Provide a detailed academic breakdown of {q.lower().rstrip('?')} with reference to {ans}."
    })

print(f"✓ Prepared {len(sum_records)} summarization and {len(para_records)} paraphrasing training examples.")

### Step 3: Fine-Tune FLAN-T5 Base (Summarization & Paraphrasing)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
import evaluate

def train_seq2seq(base_model_name, records, output_dir, metric_name, max_in_len=512, max_out_len=150):
    print(f"\n=== Fine-Tuning {output_dir} ===")
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
    
    split = int(len(records) * 0.9)
    def prep(examples):
        m_in = tokenizer(examples["input"], max_length=max_in_len, truncation=True, padding="max_length")
        lbl = tokenizer(text_target=examples["output"], max_length=max_out_len, truncation=True, padding="max_length")
        m_in["labels"] = lbl["input_ids"]
        return m_in
    
    train_ds = Dataset.from_list(records[:split]).map(prep, batched=True)
    eval_ds = Dataset.from_list(records[split:]).map(prep, batched=True)
    metric = evaluate.load(metric_name)
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)
    
    args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8 if torch.cuda.is_available() else 2,
        per_device_eval_batch_size=8 if torch.cuda.is_available() else 2,
        learning_rate=5e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
        logging_steps=10,
        report_to="none"
    )
    
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        dec_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = [[l for l in label if l != -100] for label in labels]
        dec_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        if metric_name == "rouge":
            res = metric.compute(predictions=dec_preds, references=dec_labels, use_stemmer=True)
            return {k: round(v * 100, 4) for k, v in res.items()}
        else:
            res = metric.compute(predictions=dec_preds, references=[[r] for r in dec_labels])
            return {"bleu": round(res["bleu"] * 100, 4)}
            
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics
    )
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✓ Saved fine-tuned model to {output_dir}")

# Train Summarizer
train_seq2seq("google/flan-t5-base", sum_records, "./flan-t5-study-summarizer", "rouge", 512, 150)

# Train Paraphraser
train_seq2seq("google/flan-t5-base", para_records, "./flan-t5-study-paraphraser", "sacrebleu", 256, 256)

### Step 4: Upload Fine-Tuned Models to Hugging Face Hub (Optional)

In [ ]:
from huggingface_hub import HfApi, login

# Run login() to authenticate with your Hugging Face account
# login()
# api = HfApi()
# api.upload_folder(folder_path="./flan-t5-study-summarizer", repo_id="YOUR_USERNAME/flan-t5-study-summarizer", repo_type="model")
# api.upload_folder(folder_path="./flan-t5-study-paraphraser", repo_id="YOUR_USERNAME/flan-t5-study-paraphraser", repo_type="model")